# Create Account — Interactive Notebook

Companion notebook for [`create_account.robot`](./create_account.robot).
Each section mirrors **one row** of the Robot test's `[Template]` block, but uses
raw `requests` calls against the SnapLogic API so you can see exactly what the
Robot keyword `Create Account From Template` does behind the scenes.

🔍 **Robot vs Notebook** — The Robot keyword wraps ~80 lines of Python (template
rendering, auth, idempotency check, class-FQID lookup). This notebook is a
**simplified educational version** showing the core flow. For production tests,
use the Robot keyword.

## What this notebook covers

1. **Setup** — load env vars, build a SnapLogic API session
2. **Helpers** — render template, list / check / create / delete account
3. **Row 1** — Hard-coded path / payload / name (no overwrite)
4. **Row 2** — Same payload, different name, with `overwrite_if_exists=True`
5. **Row 3** — All values from env vars (the realistic pattern)
6. **Verify** — list accounts to confirm what was created
7. **Cleanup** — delete the test accounts (optional)

## Prerequisites

- Start Jupyter via `make jupyter-start` (or `make start-services`).
- Required env vars (auto-loaded by the Setup cell from `/app/.env` and `/app/env_files/database_accounts/.env.sqlserver`):
  - `URL`, `ORG_ADMIN_USER`, `ORG_ADMIN_PASSWORD`, `ORG_NAME`
  - `ACCOUNT_LOCATION_PATH`
  - `SQLSERVER_ACCOUNT_PAYLOAD_FILE_NAME`, `SQLSERVER_ACCOUNT_NAME`
  - All `SQLSERVER_*` connection vars (`HOST`, `PORT`, `USER`, `PASSWORD`, `DATABASE`)

## Section 1 — Setup

Load credentials, validate they're present, and create a `requests.Session` with Basic Auth.

In [16]:
import os
import json
from pathlib import Path
import requests
from jinja2 import Environment, StrictUndefined
from dotenv import load_dotenv

# --- Load env vars from root .env + database-specific env files ---------------
# Root .env is already injected by Docker Compose (env_file:) but reloading is
# harmless. Database-specific env files (env_files/.../*.env.sqlserver etc.)
# are NOT auto-loaded by Compose, so we load them here.
load_dotenv("/app/.env", override=False)
load_dotenv("/app/env_files/database_accounts/.env.sqlserver", override=False)

# --- SnapLogic API credentials -------------------------------------------------
SL_URL  = os.environ.get("URL",                "https://elastic.snaplogic.com")
SL_USER = os.environ.get("ORG_ADMIN_USER")
SL_PASS = os.environ.get("ORG_ADMIN_PASSWORD")
ORG     = os.environ.get("ORG_NAME")

# Sanity check
missing = [k for k, v in {"ORG_ADMIN_USER": SL_USER, "ORG_ADMIN_PASSWORD": SL_PASS, "ORG_NAME": ORG}.items() if not v]
if missing:
    raise RuntimeError(f"Missing required env vars: {missing}. Source your .env files first.")

# --- Reusable session with Basic auth ------------------------------------------
session = requests.Session()
session.auth = (SL_USER, SL_PASS)
session.headers.update({"Content-Type": "application/json"})

print(f"✅ SnapLogic session ready")
print(f"   URL: {SL_URL}")
print(f"   Org: {ORG}")
print(f"   User: {SL_USER}")

✅ SnapLogic session ready
   URL: https://cdn.elastic.snaplogic.com
   Org: ml-legacy-migration
   User: spothana@snaplogic.com


## Section 2 — Helper functions

These mirror what `Create Account From Template` does internally:

1. **`render_template`** — load JSON file, render Jinja2 placeholders from `os.environ`, inject `asset_type: "Account"`
2. **`list_accounts`** — list account assets under a project path (uses `/asset/list/...` endpoint)
3. **`account_exists`** — check if a named account already exists
4. **`create_account`** — POST the rendered payload to the SnapLogic API
5. **`delete_account`** — DELETE an existing account by path + name
6. **`create_or_overwrite`** — top-level orchestration that mirrors the Robot keyword

In [17]:
# Path to checked-in account JSON templates (inside the tools container)
ACCOUNT_PAYLOAD_DIR = Path("/app/test/suite/test_data/accounts_payload")

# Jinja2 environment — matches what the Robot keyword uses (handles
# `{{ VAR }}` plus filters like `{{ VAR | default('[]', true) }}`).
_jinja_env = Environment(keep_trailing_newline=True)


def render_template(template_filename: str, override_account_name: str | None = None) -> dict:
    """
    Load a JSON template file and render Jinja2 placeholders using os.environ as context.
    Injects `asset_type: "Account"` at the top level (required by SnapLogic create_asset API).
    Optionally override the account name embedded in the template.
    """
    raw = (ACCOUNT_PAYLOAD_DIR / template_filename).read_text()
    rendered = _jinja_env.from_string(raw).render(**os.environ)
    payload = json.loads(rendered)
    payload["asset_type"] = "Account"
    if override_account_name:
        payload["account"]["property_map"]["info"]["label"]["value"] = override_account_name
    return payload


def list_accounts(account_location_path: str) -> list:
    """List all Account assets under <org>/<location_path>.

    Uses the /asset/list/ endpoint — the plain /asset/{org}/{path} endpoint
    returns metadata about the directory itself, not its children.
    """
    url = f"{SL_URL}/api/1/rest/asset/list/{ORG}/{account_location_path}"
    resp = session.get(url)
    resp.raise_for_status()
    entries = resp.json().get("response_map", {}).get("entries", [])
    return [e for e in entries if e.get("asset_type") == "Account"]


def account_exists(account_location_path: str, account_name: str) -> dict | None:
    """Return the matching account entry if it exists, else None."""
    for entry in list_accounts(account_location_path):
        if entry["name"] == account_name:
            return entry
    return None


def create_account(account_location_path: str, payload: dict) -> dict:
    """POST the rendered payload to the SnapLogic API; return the response JSON."""
    account_name = payload["account"]["property_map"]["info"]["label"]["value"]
    url = f"{SL_URL}/api/1/rest/asset/{ORG}/{account_location_path}/{account_name}"
    resp = session.post(url, json=payload)
    if not resp.ok:
        print(f"❌ {resp.status_code} response body: {resp.text[:1000]}")
    resp.raise_for_status()
    return resp.json()


def delete_account(account_location_path: str, account_name: str) -> None:
    """DELETE an account using its path-based URL (asset_id-based URL returns 404)."""
    url = f"{SL_URL}/api/1/rest/asset/{ORG}/{account_location_path}/{account_name}"
    resp = session.delete(url)
    resp.raise_for_status()


def create_or_overwrite(
    account_location_path: str,
    template_filename: str,
    account_name: str,
    overwrite_if_exists: bool = False,
) -> str:
    """
    Mirrors the Robot keyword `Create Account From Template`.
    Returns one of: 'created', 'overwritten', 'skipped'.
    """
    print(f"📄 Loading template:  {template_filename}")
    payload = render_template(template_filename, override_account_name=account_name)
    print(f"🔍 Account name:      {account_name}")
    print(f"🔍 Org / Path:        {ORG} / {account_location_path}")

    existing = account_exists(account_location_path, account_name)
    if existing:
        if overwrite_if_exists:
            print(f"🗑️  Account exists — deleting to overwrite")
            delete_account(account_location_path, account_name)
            create_account(account_location_path, payload)
            print(f"✅ Account [{account_name}] overwritten")
            return "overwritten"
        else:
            print(f"ℹ️  Account [{account_name}] already exists — skipping (overwrite_if_exists=False)")
            return "skipped"

    create_account(account_location_path, payload)
    print(f"✅ Account [{account_name}] created")
    return "created"

print("✅ Helper functions defined")

✅ Helper functions defined


## Section 3 — Row 1: Hard-coded path / payload / name (no overwrite)

Mirrors this Robot test row:

```robot
test10_project_space/shared    acc_sqlserver.json    sqlserveracct3
```

First call → account is created.
Re-run this cell → account already exists, gets skipped (no error).


In [21]:
result = create_or_overwrite(
    account_location_path="test10_project_space/shared",
    template_filename="acc_sqlserver.json",
    account_name="sqlserveracct3",
    overwrite_if_exists=False,
)
print(f"\nResult: {result}")

📄 Loading template:  acc_sqlserver.json
🔍 Account name:      sqlserveracct3
🔍 Org / Path:        ml-legacy-migration / test10_project_space/shared
ℹ️  Account [sqlserveracct3] already exists — skipping (overwrite_if_exists=False)

Result: skipped


## Section 4 — Row 2: Same payload, different name, with `overwrite_if_exists=True`

Mirrors this Robot test row:

```robot
test10_project_space/shared    acc_sqlserver.json    sqlserveracct4    overwrite_if_exists=${TRUE}
```

First call → account is created.
Re-run this cell → account exists, gets **deleted and recreated** (overwrite path).


In [22]:
result = create_or_overwrite(
    account_location_path="test10_project_space/shared",
    template_filename="acc_sqlserver.json",
    account_name="sqlserveracct4",
    overwrite_if_exists=True,
)
print(f"\nResult: {result}")

📄 Loading template:  acc_sqlserver.json
🔍 Account name:      sqlserveracct4
🔍 Org / Path:        ml-legacy-migration / test10_project_space/shared
🗑️  Account exists — deleting to overwrite
✅ Account [sqlserveracct4] overwritten

Result: overwritten


## Section 5 — Row 3: All values from env vars (the realistic pattern)

Mirrors this Robot test row:

```robot
${ACCOUNT_LOCATION_PATH}    ${SQLSERVER_ACCOUNT_PAYLOAD_FILE_NAME}    ${SQLSERVER_ACCOUNT_NAME}
```

Uses values from your `.env` and `env_files/database_accounts/.env.sqlserver` —
no hard-coded strings in this cell.


In [20]:
ACCOUNT_LOCATION_PATH        = os.environ["ACCOUNT_LOCATION_PATH"]
SQLSERVER_PAYLOAD_FILE_NAME  = os.environ["SQLSERVER_ACCOUNT_PAYLOAD_FILE_NAME"]
SQLSERVER_ACCOUNT_NAME       = os.environ["SQLSERVER_ACCOUNT_NAME"]

print(f"   ACCOUNT_LOCATION_PATH:               {ACCOUNT_LOCATION_PATH}")
print(f"   SQLSERVER_ACCOUNT_PAYLOAD_FILE_NAME: {SQLSERVER_PAYLOAD_FILE_NAME}")
print(f"   SQLSERVER_ACCOUNT_NAME:              {SQLSERVER_ACCOUNT_NAME}")
print()

result = create_or_overwrite(
    account_location_path=ACCOUNT_LOCATION_PATH,
    template_filename=SQLSERVER_PAYLOAD_FILE_NAME,
    account_name=SQLSERVER_ACCOUNT_NAME,
    overwrite_if_exists=False,
)
print(f"\nResult: {result}")

   ACCOUNT_LOCATION_PATH:               test10_project_space/shared
   SQLSERVER_ACCOUNT_PAYLOAD_FILE_NAME: acc_sqlserver.json
   SQLSERVER_ACCOUNT_NAME:              sqlserver_acct

📄 Loading template:  acc_sqlserver.json
🔍 Account name:      sqlserver_acct
🔍 Org / Path:        ml-legacy-migration / test10_project_space/shared
ℹ️  Account [sqlserver_acct] already exists — skipping (overwrite_if_exists=False)

Result: skipped


## Section 6 — List all accounts (verify what was created)

Useful sanity check — confirms the previous cells created what they claim.


In [ ]:
for path in {"test10_project_space/shared", ACCOUNT_LOCATION_PATH}:
    print(f"📋 Accounts under {ORG}/{path}:")
    for entry in list_accounts(path):
        asset_id = (entry.get("asset_id") or "")[:16]
        print(f"   - {entry['name']:30s}  ({asset_id}...)")
    print()

## Section 7 — Cleanup (optional)

Delete the test accounts created above so the project space is left clean.
Skip this if you want to inspect the accounts in the SnapLogic Manager UI.


In [23]:
TO_CLEAN = [
    ("test10_project_space/shared", "sqlserveracct3"),
    ("test10_project_space/shared", "sqlserveracct4"),
    (ACCOUNT_LOCATION_PATH,          SQLSERVER_ACCOUNT_NAME),
]

for path, name in TO_CLEAN:
    if account_exists(path, name):
        delete_account(path, name)
        print(f"🗑️  deleted {path}/{name}")
    else:
        print(f"ℹ️  {path}/{name} not found — nothing to delete")

🗑️  deleted test10_project_space/shared/sqlserveracct3
🗑️  deleted test10_project_space/shared/sqlserveracct4
🗑️  deleted test10_project_space/shared/sqlserver_acct


## What's next

- **Tweak any cell** — change account name, switch templates (try `acc_oracle.json` or `acc_postgres.json`), set `overwrite_if_exists`.
- **Inspect in SnapLogic Manager** — log into the SnapLogic Manager UI to see the accounts you just created.
- **Run the official Robot test instead** for production validation:
  ```bash
  make robot-run-tests-no-gp TAGS="create_account_sample"
  ```

## Caveats vs the Robot keyword

This notebook is a simplified educational version. The Robot keyword `Create Account From Template`
additionally:

- Auto-detects auth mode (Basic vs OAuth) and refreshes tokens
- Looks up the correct `class_fqid` version from a snap-pack registry
- Handles paged project listings (>100 entries)
- Provides Robot-style logging, tags, and Suite Variable side-effects (`${account_name}_snodeid` etc.)

For production tests, **use the Robot keyword**. This notebook is for learning what's underneath.

## Related

- [`create_account.robot`](./create_account.robot) — runnable Robot test (3 rows)
- [`create_account_explained.md`](./create_account_explained.md) — manual vs automation walkthrough
- [`s3_connection_operations.ipynb`](../../02.verification_test_cases/02.%20connect_to_s3_operations/s3_connection_operations.ipynb) — sibling notebook for S3 ops